In [0]:
USE vendas_pecas.camada_ouro;

In [0]:
CREATE OR REPLACE TEMP VIEW vw_vendas
AS
SELECT *
FROM 
  vendas_pecas.camada_prata.tb_prata tp
WHERE 
  tp.IdNotaFiscal IS NOT NULL
  -- tp.quantidade IS NOT NULL and
  -- tp.quantidade > 0 and
  -- tp.valor_unitario > 0 and
  -- tp.custo_unitario > 0

##Tabelas Dimensão

In [0]:
USE vendas_pecas.camada_ouro;

In [0]:
--merge
CREATE OR REPLACE TABLE tb_dim_produtos AS
SELECT
  ROW_NUMBER() OVER (ORDER BY produto_peca_sem_acento) as produto_id,
  produto_peca_sem_acento
FROM 
  (SELECT DISTINCT (produto_peca_sem_acento) FROM vw_vendas)
ORDER BY (produto_peca_sem_acento)
       

In [0]:
CREATE OR REPLACE TABLE tb_dim_cliente AS
SELECT
  cliente_id,
  cliente_nome
FROM
  (SELECT DISTINCT
  cliente_id,
  cliente_nome FROM vw_vendas)

  

In [0]:
CREATE OR REPLACE TABLE tb_dim_marca_carro AS
SELECT 
  row_number() OVER (ORDER BY marca_carro) AS marca_carro_id,
  marca_carro
FROM
  (SELECT DISTINCT
  marca_carro FROM vw_vendas)

In [0]:
CREATE OR REPLACE TABLE tb_dim_modelo_carro AS
SELECT
  ROW_NUMBER() OVER (ORDER BY modelo_carro) AS modelo_carro_id,
  modelo_carro
FROM
  (SELECT DISTINCT modelo_carro FROM vw_vendas)

In [0]:
CREATE OR REPLACE TABLE tb_dim_carro AS
SELECT
  ROW_NUMBER() OVER (order by c.modelo_carro) AS carro_id,
  modelo_carro_id,
  marca_carro_id
FROM (SELECT DISTINCT modelo_carro, marca_carro FROM vw_vendas) c
JOIN tb_dim_modelo_carro mc
ON c.modelo_carro = mc.modelo_carro
JOIN tb_dim_marca_carro m
ON c.marca_carro = m.marca_carro

In [0]:
CREATE OR REPLACE TABLE vendas_pecas.camada_ouro.tb_dim_vendedor AS
SELECT DISTINCT
  vendedor_id,
  vendedor_nome
FROM
  vw_vendas

In [0]:
CREATE OR REPLACE TABLE vendas_pecas.camada_ouro.tb_dim_loja AS
SELECT DISTINCT
  loja_id,
  loja_nome_sem_acento,
  grupo_loja
FROM vw_vendas
order by grupo_loja

In [0]:
CREATE OR REPLACE TABLE vendas_pecas.camada_ouro.tb_dim_data_venda AS
SELECT
  ROW_NUMBER() OVER (ORDER BY data_venda) AS data_venda_id,
  data_venda, ano_venda,  mes_venda, DAY(data_venda) as dia_venda
FROM (SELECT DISTINCT data_venda, ano_venda, mes_venda FROM vw_vendas)



##Tabela Fato

In [0]:
CREATE OR REPLACE TABLE vendas_pecas.camada_ouro.tb_fato_vendas AS
SELECT
  fv.IdNotaFiscal,
  fv.valor_unitario,
  fv.quantidade,
  fv.custo_unitario,
  dcl.cliente_id,
  dc.carro_id,
  ddv.data_venda_id,
  dl.loja_id,
  dpd.produto_id,
  dv.vendedor_id,
  try_cast((fv.valor_unitario * fv.quantidade) as decimal (18,2)) as faturamento,
  try_cast((faturamento - (fv.custo_unitario * fv.quantidade)) as decimal(18,2)) as lucro,
  file_path,
  ingest_date
FROM vw_vendas fv
LEFT JOIN tb_dim_cliente dcl ON fv.cliente_id = dcl.cliente_id
LEFT JOIN tb_dim_data_venda ddv ON fv.data_venda = ddv.data_venda
LEFT JOIN tb_dim_loja dl ON fv.loja_nome_sem_acento = dl.loja_nome_sem_acento
LEFT JOIN tb_dim_produtos dpd ON fv.produto_peca_sem_acento = dpd.produto_peca_sem_acento
LEFT JOIN tb_dim_vendedor dv ON fv.vendedor_nome = dv.vendedor_nome
LEFT JOIN tb_dim_modelo_carro md  ON fv.modelo_carro = md.modelo_carro 
LEFT JOIN tb_dim_marca_carro mc ON fv.marca_carro = mc.marca_carro
LEFT JOIN tb_dim_carro dc ON dc.modelo_carro_id = md.modelo_carro_id and dc.marca_carro_id = mc.marca_carro_id
ORDER BY IdNotaFiscal 
                                            

##Consultas

####Quais períodos apresentam melhor e pior performance?

In [0]:
USE vendas_pecas.camada_ouro;

In [0]:
SELECT 
  ano_venda,
  CASE
    WHEN mes_venda <= 6 
      THEN '1º Semestre'
    WHEN mes_venda > 6 
      THEN '2º Semestre'
  END as Semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt ON fv.data_venda_id = ddt.data_venda_id
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  ano_venda, semestre
ORDER BY 
  ano_venda


In [0]:
SELECT 
    ano_venda, 
    mes_venda, 
    SUM(faturamento) AS faturamento, 
    SUM(lucro) as lucro
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt on fv.data_venda_id = ddt.data_venda_id
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY  
  ano_venda, mes_venda
ORDER BY 
  ano_venda, mes_venda

In [0]:
SELECT 
  ano_venda,
  mes_venda,
  CASE
    WHEN day(data_venda) <= 15 
      THEN '1º Quinzena'
    WHEN day(data_venda) > 15 
      THEN '2º Quinzena'
  END AS Semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt ON fv.data_venda_id = ddt.data_venda_id
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  ano_venda, mes_venda, semestre
ORDER BY 
  ano_venda,mes_venda


####Qual o desempenho das vendas por loja, grupo de lojas e vendedor?

In [0]:
SELECT 
  grupo_loja AS grupo_loja,
  loja_nome_sem_acento AS loja_nome, 
  SUM(faturamento) AS faturamento, 
  SUM(lucro) AS lucro,
  SUM(faturamento) / count(IdNotaFiscal) AS ticket_medio
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON fv.loja_id = dl.loja_id
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
   grupo_loja, loja_nome
ORDER BY 
  faturamento desc

In [0]:
SELECT
  vendedor_nome,
  SUM(faturamento) AS faturamento,
  SUM(lucro)AS lucro,
  SUM(faturamento) / count(IdNotaFiscal) AS ticket_medio
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_vendedor dv ON fv.vendedor_id = dv.vendedor_id
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0  and
  dv.vendedor_id = fv.vendedor_id

GROUP BY 
  vendedor_nome
ORDER BY 
  faturamento DESC
   


####Quais unidades e grupos concentram maior volume de vendas?


In [0]:
SELECT 
  grupo_loja,
  loja_nome_sem_acento AS loja_nome,
  count(IdNotaFiscal) AS n_vendas,
  SUM(quantidade) AS qtd_vendida
FROM
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON dl.loja_id = fv.loja_id
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0  
GROUP BY 
  loja_nome_sem_acento, grupo_loja
ORDER BY n_vendas DESC


In [0]:
SELECT 
  grupo_loja,
  count(IdNotaFiscal) AS n_vendas,
  SUM(quantidade) AS qtd_vendida
FROM
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON dl.loja_id = fv.loja_id
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  grupo_loja
ORDER BY n_vendas DESC


####Como está o comparativo entre anos?

In [0]:
SELECT
    ano_venda AS ano,
    SUM(faturamento) AS faturamento, 
    SUM(lucro) AS lucro,  
    SUM(quantidade) AS qtd_pecas_vendidas,
    COUNT(IdNotaFiscal) AS n_vendas,
    ROW_NUMBER() OVER(
      PARTITION BY ano_venda
      ORDER BY sum(faturamento) DESC
  ) AS rn_quantidade
  FROM 
    tb_fato_vendas fv
  JOIN 
    tb_dim_data_venda ddt ON ddt.data_venda_id = fv.data_venda_id
  JOIN
    tb_dim_produtos dp ON dp.produto_id = fv.produto_id
  JOIN
    tb_dim_loja dl ON dl.loja_id = fv.loja_id
  WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
  GROUP BY  ano